# Notebook 02: Data Preparation

A model is only ever as good as what you feed it. This notebook is about turning a pile of messy documents into clean training material the model can actually learn from. It is less glamorous than training, but it is where most of the real work in any AI project lives.

## The problem we are solving

The base model knows the world in general. To be useful to a specific organisation it needs to know their things:

- Their products, clients, and in-house jargon
- Their actual documents: contracts, reports, policies, research
- Their style: how their analysts write, how their reports are laid out

That information starts life as unstructured files: PDFs, web pages, plain text. You cannot pour those straight into training. They have to be cleaned up and reshaped first, and that is what we do here.

The steps:

1. Download some public financial documents (filings from the SEC).
2. Clean them and cut them into bite-sized chunks.
3. Build an "instruction dataset", the question-and-answer format used for fine-tuning.
4. Generate extra training examples automatically, using a model to help.

## Adapt this

We use SEC filings because they are public and free. To point this at your own documents, just change how you load them at the start:

- PDFs: pull the text out with a library like `pypdf`, then carry on from the cleaning step.
- Internal wikis or Confluence: export to text or markdown and load the files.
- Web pages: extract the readable text with a tool like `trafilatura` or BeautifulSoup.
- Database records: pull them into a table and turn each row into a sentence.

Everything after the loading step is exactly the same no matter where the documents came from.


In [ ]:
import os
import re
import json
import random
import textwrap
from pathlib import Path

import pandas as pd
from datasets import Dataset

# Set seeds for reproducibility - important when generating datasets
random.seed(42)

# Paths - all data goes into the data/ directory at the project root
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Directories ready.")

## 1. Downloading filings from the SEC

The SEC is the US financial regulator. It requires public companies to file regular reports, and it makes them free for anyone to read through a system called EDGAR. That makes them a great stand-in for a company's private documents: realistic, detailed, and legal to share.

The three filings worth knowing:

- **10-K**: the annual report. The big one, with full financial results, risks, and a business overview.
- **10-Q**: a shorter quarterly update.
- **8-K**: a one-off notice when something important happens, like an earnings release.

We pull a couple of 10-K annual reports in the next cell.


In [ ]:
# We use the sec-edgar-downloader library to pull filings programmatically.
# SEC requires a user agent string (your name/email) - this is a public API.

from sec_edgar_downloader import Downloader

# Initialise downloader
# Replace with your own name/email (SEC requirement, no authentication needed)
dl = Downloader("LLM Adaptation Project", "research@example.com", str(RAW_DIR))

# Download 10-K filings for a few large public companies
# limit=2 means we get the 2 most recent annual reports per company
companies = {
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "JPM": "JPMorgan Chase",
}

for ticker, name in companies.items():
    print(f"Downloading {name} ({ticker}) annual report...")
    dl.get("10-K", ticker, limit=2)

print("\nDownload complete. Files saved to:", RAW_DIR)

In [ ]:
# Let's see what we downloaded
import os

all_files = list(RAW_DIR.rglob("*.txt")) + list(RAW_DIR.rglob("*.htm"))
print(f"Found {len(all_files)} files")
for f in all_files[:10]:  # show first 10
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:50s}  {size_kb:8.0f} KB")

## 2. Cleaning the raw text

Filings come as HTML or plain text and they are full of junk: HTML tags, legal boilerplate, repeated headers, page numbers, stray whitespace. None of that helps the model, and some of it actively hurts. So before we do anything else, we strip it out and keep just the readable content.


In [ ]:
def clean_text(raw_text: str) -> str:
    """
    Clean raw SEC filing text.
    Steps:
      1. Strip HTML tags if present
      2. Normalise whitespace
      3. Remove very short lines (page numbers, stray chars)
      4. Collapse repeated blank lines
    """
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', raw_text)
    
    # Decode common HTML entities
    text = text.replace('&amp;', '&').replace('&nbsp;', ' ').replace('&lt;', '<').replace('&gt;', '>')
    
    # Normalise whitespace within lines
    lines = text.split('\n')
    lines = [' '.join(line.split()) for line in lines]
    
    # Drop lines that are too short to be meaningful (e.g. page numbers, lone symbols)
    lines = [line for line in lines if len(line) > 40]
    
    # Rejoin
    text = '\n'.join(lines)
    
    # Collapse multiple blank lines into one
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()


# Load and clean one filing as a demo
sample_file = next(RAW_DIR.rglob("*.txt"), None) or next(RAW_DIR.rglob("*.htm"), None)

if sample_file:
    raw = sample_file.read_text(encoding="utf-8", errors="ignore")
    cleaned = clean_text(raw)
    
    print(f"File: {sample_file.name}")
    print(f"Raw length   : {len(raw):,} characters")
    print(f"Cleaned length: {len(cleaned):,} characters")
    print(f"Reduction    : {(1 - len(cleaned)/len(raw))*100:.0f}%")
    print()
    print("--- First 500 characters of cleaned text ---")
    print(cleaned[:500])
else:
    print("No files found - check the download step above.")

## 3. Chunking

A 10-K can run to hundreds of pages, and a model can only read so much at once. That limit is called the context window. So we slice each document into smaller chunks that fit.

Two choices matter when chunking:

- **Chunk size.** Too small and a chunk loses its surrounding context. Too big and it will not fit in the window. We aim for a sensible middle.
- **Overlap.** We let each chunk share a little text with the one before it. That way a sentence sitting on the boundary between two chunks does not get cut in half and lost.


In [ ]:
def chunk_text(text: str, chunk_size: int = 512, overlap: int = 64) -> list[str]:
    """
    Split text into overlapping chunks by word count.
    
    chunk_size: target number of words per chunk
    overlap: number of words shared between consecutive chunks
    """
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # move forward by (chunk_size - overlap)
    
    return chunks


# Apply to our cleaned text
if sample_file:
    chunks = chunk_text(cleaned, chunk_size=512, overlap=64)
    
    print(f"Document split into {len(chunks)} chunks")
    print(f"Average chunk size: {sum(len(c.split()) for c in chunks) / len(chunks):.0f} words")
    print()
    print("--- Example chunk ---")
    print(textwrap.fill(chunks[5], width=80))

## 4. Building an instruction dataset

For fine-tuning, the model does not learn from loose text. It learns from examples laid out as an instruction and a response, often with some supporting context. One example looks like this:

```
### Instruction:
What were Apple's main sources of revenue last year?

### Context:
Apple reported total net sales of $383.3 billion for fiscal 2023. Products revenue
was $298.1 billion, driven by iPhone at $200.6 billion...

### Response:
Apple's main revenue sources were Products ($298.1B) and Services ($85.2B)...
```

Laying the data out this way teaches the model two habits at once: follow the instruction it is given, and base its answer on the context provided rather than on whatever it half-remembers. That is exactly the behaviour we want from a document assistant.


In [ ]:
import sys
sys.path.append("..")
from utils.templates import format_instruction

# For demonstration, we'll create a small handcrafted dataset based on real SEC data.
# In production, you'd generate these at scale using the synthetic approach in the next section.

manual_examples = [
    {
        "instruction": "What was Apple's total net sales for fiscal year 2023?",
        "context": "Apple Inc. reported total net sales of $383.3 billion for the fiscal year ended September 30, 2023, compared to $394.3 billion for fiscal 2022, representing a decrease of 2.8%.",
        "response": "Apple's total net sales for fiscal year 2023 were $383.3 billion, a decrease of 2.8% from $394.3 billion in fiscal 2022."
    },
    {
        "instruction": "What are the main risks Apple identifies in its annual report?",
        "context": "The Company's operations and financial results are subject to various risks and uncertainties, including global economic conditions, intense competition, rapid technological change, dependence on component suppliers, and cybersecurity threats.",
        "response": "Apple identifies several key risks in its annual report: global economic conditions affecting consumer spending, intense competition in its product markets, rapid technological change requiring continuous innovation, supplier concentration risk, and cybersecurity and data privacy threats."
    },
    {
        "instruction": "How does JPMorgan Chase describe its credit risk management approach?",
        "context": "JPMorgan Chase manages credit risk through diversification, credit limits, collateral requirements, and credit derivatives. The firm uses internal credit ratings and stress testing to assess portfolio quality.",
        "response": "JPMorgan Chase manages credit risk through four main mechanisms: portfolio diversification to avoid concentration, credit exposure limits per counterparty, collateral requirements to secure exposures, and hedging via credit derivatives. The firm supplements these controls with internal credit ratings and stress testing."
    },
    {
        "instruction": "What does Microsoft report as its three main business segments?",
        "context": "Microsoft operates through three segments: Productivity and Business Processes (including Office and LinkedIn), Intelligent Cloud (including Azure and server products), and More Personal Computing (including Windows and Xbox).",
        "response": "Microsoft's three business segments are: (1) Productivity and Business Processes, which includes Office 365, LinkedIn, and Dynamics; (2) Intelligent Cloud, anchored by Azure cloud services and server products; and (3) More Personal Computing, covering Windows, Xbox, and Surface devices."
    },
    {
        "instruction": "Summarise the key financial performance metrics for Microsoft in fiscal 2023.",
        "context": "Microsoft reported revenue of $211.9 billion for fiscal year 2023, up 7% year-over-year. Operating income was $88.5 billion (+6%), and net income was $72.4 billion (+0.4%). Azure and cloud services grew 27%.",
        "response": "Microsoft's fiscal 2023 financial highlights: revenue of $211.9B (+7% YoY), operating income of $88.5B (+6%), and net income of $72.4B (roughly flat). The standout performer was Azure and cloud services, which grew 27% and demonstrates the ongoing shift to Microsoft's cloud business."
    },
]

# Format each example using the instruction template
formatted = [
    format_instruction(ex["instruction"], ex["response"], ex["context"])
    for ex in manual_examples
]

print("Example formatted instruction:")
print("-" * 60)
print(formatted[0])

## 5. Generating training data automatically

Writing enough question-and-answer pairs by hand would take weeks. There is a much faster way: let a model write them for you.

The idea is simple:

1. Take a chunk of one of your documents.
2. Ask a capable model to read it and come up with a question and a matching answer.
3. Keep the good pairs and use them as training data for your own smaller model.

This is often called using a model "as a teacher". It scales nicely, because once it works on one chunk it works on thousands.

A practical note: in a real project you would use a strong model like Claude or GPT-4 through an API for this step, since the quality of the teacher matters. Here we use the small local model instead, just to show the pattern, so you do not need an API key to run it.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokeniser = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Build a text generation pipeline - a convenient wrapper around tokenise -> generate -> decode
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokeniser,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.8,
)

print("Model ready for synthetic data generation.")

In [ ]:
def generate_qa_from_chunk(chunk: str, generator) -> dict | None:
    """
    Given a document chunk, ask the model to generate a question and answer.
    Returns a dict with keys: instruction, context, response
    """
    prompt = f"""Based on the following financial text, generate one specific question 
and a concise answer. Format your response as:
Q: [question]
A: [answer]

Text: {chunk[:400]}

Q:"""
    
    output = generator(prompt)[0]["generated_text"]
    generated = output[len(prompt):].strip()
    
    # Parse out Q and A from the generated text
    if "\nA:" in generated:
        parts = generated.split("\nA:", 1)
        question = parts[0].strip().lstrip("Q:").strip()
        answer = parts[1].strip().split("\n")[0].strip()
        
        if len(question) > 20 and len(answer) > 20:
            return {
                "instruction": question,
                "context": chunk[:400],
                "response": answer,
            }
    return None


# Generate a few synthetic examples from our document chunks
synthetic_examples = []
example_chunks = chunks[:5] if 'chunks' in dir() else []

if example_chunks:
    print("Generating synthetic Q&A pairs from document chunks...\n")
    for i, chunk in enumerate(example_chunks):
        result = generate_qa_from_chunk(chunk, generator)
        if result:
            synthetic_examples.append(result)
            print(f"Example {i+1}:")
            print(f"  Q: {result['instruction']}")
            print(f"  A: {result['response'][:100]}...")
            print()
    print(f"Generated {len(synthetic_examples)} valid examples from {len(example_chunks)} chunks.")
else:
    print("No chunks available - using the manual examples from earlier.")
    synthetic_examples = manual_examples

## 6. Saving the dataset

Finally we bundle the hand-written and auto-generated examples together and save them in the standard Hugging Face dataset format, which is what the training notebooks expect. We also split off a small slice for validation, so later we can check the model on examples it did not train on.


In [ ]:
# Combine manual and synthetic examples
all_examples = manual_examples + synthetic_examples

# Add formatted text column (the instruction template applied to each example)
for ex in all_examples:
    ex["text"] = format_instruction(ex["instruction"], ex["response"], ex.get("context"))

# Convert to HuggingFace Dataset
dataset = Dataset.from_list(all_examples)

# Split into train / validation
# 90% train, 10% validation - standard practice
split = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Total examples : {len(dataset)}")
print(f"Train split    : {len(split['train'])}")
print(f"Val split      : {len(split['test'])}")
print()
print("Dataset schema:")
print(dataset.features)

In [ ]:
# Save to disk in a format compatible with the rest of the workflow
split.save_to_disk(str(PROCESSED_DIR / "instruction_dataset"))

# Also save a JSON version for easy inspection
with open(PROCESSED_DIR / "examples.json", "w") as f:
    json.dump(all_examples, f, indent=2)

print(f"Dataset saved to {PROCESSED_DIR}")
print()
print("Files created:")
for p in PROCESSED_DIR.rglob("*"):
    if p.is_file():
        print(f"  {p.relative_to(PROCESSED_DIR)}")

## Recap

In this notebook we:

- Downloaded real public financial filings from the SEC
- Cleaned the raw HTML and text down to the useful content
- Cut long documents into overlapping chunks the model can handle
- Built an instruction dataset in the question-and-answer format fine-tuning needs
- Used a model to generate extra training examples, the same trick used in serious dataset pipelines
- Saved everything in the standard format the next notebooks read

We now have what we need to take either of the two adaptation paths:

- Notebook 03 uses the document chunks to build a search-and-retrieve (RAG) system.
- Notebook 04 uses the instruction dataset to fine-tune the model.

Next up: notebook 03, the RAG pipeline.
